In [6]:
import os
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

from PIL import Image

In [7]:
TRAIN_IMG = "./NEU-DET/train/images"
VAL_IMG = "./NEU-DET/validation/images"
TRAIN_ANN = "./NEU-DET/train/annotations"
VAL_ANN = "./NEU-DET/validation/annotations"
OUTPUT_TRAIN_IMG = "./Dataset/images/train"
OUTPUT_TRAIN_LBL = "./Dataset/labels/train"
OUTPUT_VAL_IMG = "./Dataset/images/validation"
OUTPUT_VAL_LBL = "./Dataset/labels/validation"

# Class mapping
CLASS_MAP = {"cr": 0, "in": 1, "pa": 2, "ps": 3, "rs": 4, "sc": 5}

In [ ]:
def convert_voc_to_yolo(xml_path, img_w, img_h):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []

    for obj in root.findall("object"):
        cls_name = obj.find("name").text.lower()
        if cls_name not in CLASS_MAP:
            continue

        bndbox = obj.find("bndbox")
        xmin = int(bndbox.find("xmin").text)
        ymin = int(bndbox.find("ymin").text)
        xmax = int(bndbox.find("xmax").text)
        ymax = int(bndbox.find("ymax").text)

        x_center = ((xmin + xmax) / 2) / img_w  # Normalization
        y_center = ((ymin + ymax) / 2) / img_h
        width = (xmax - xmin) / img_w
        height = (ymax - ymin) / img_h

        lines.append(f"{CLASS_MAP[cls_name]} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

    return lines


def find_image_path(image_dir, stem):
    for extension in (".jpg", ".jpeg"):
        candidate = image_dir / f"{stem}{extension}"
        if candidate.exists():
            return candidate

    matches = list(image_dir.rglob(f"{stem}.*"))
    return matches[0] if matches else None


def convert_split(annotation_dir, image_dir, output_image_dir, output_label_dir):
    if output_image_dir.exists():
        shutil.rmtree(output_image_dir)
    if output_label_dir.exists():
        shutil.rmtree(output_label_dir)

    output_image_dir.mkdir(parents=True, exist_ok=True)
    output_label_dir.mkdir(parents=True, exist_ok=True)

    processed_count = 0
    skipped_files = []

    #parsing xml file
    for xml_path in sorted(annotation_dir.glob("*.xml")):  
        image_path = find_image_path(image_dir, xml_path.stem)
        if image_path is None:
            skipped_files.append(xml_path.name)
            continue

        with Image.open(image_path) as image_file:
            image_width, image_height = image_file.size

        yolo_lines = convert_voc_to_yolo(xml_path, image_width, image_height)
        shutil.copy2(image_path, output_image_dir / image_path.name)
        (output_label_dir / f"{xml_path.stem}.txt").write_text("".join(yolo_lines), encoding="utf-8")
        processed_count += 1

    return processed_count, skipped_files


splits = {
    "train": (Path(TRAIN_ANN), Path(TRAIN_IMG), Path(OUTPUT_TRAIN_IMG), Path(OUTPUT_TRAIN_LBL)),
    "validation": (Path(VAL_ANN), Path(VAL_IMG), Path(OUTPUT_VAL_IMG), Path(OUTPUT_VAL_LBL)),
}

summary = {}
for split_name, (annotation_dir, image_dir, output_image_dir, output_label_dir) in splits.items():
    processed_count, skipped_files = convert_split(annotation_dir, image_dir, output_image_dir, output_label_dir)
    summary[split_name] = {"processed": processed_count, "skipped": skipped_files}
    print(f"{split_name}: converted {processed_count} images")
    if skipped_files:
        print(f"{split_name}: skipped {len(skipped_files)} XML files")


train: converted 1439 images
validation: converted 360 images
validation: skipped 1 XML files without a matching image
Conversion complete.
